# EV Charging & Adoption Dashboard (2010 - 2030)

The notebook is organized as a **six-tab story**, each resolved a big question direction.

| Tab | Theme | Question it answers |
|---|---|---|
| *0. Overview* | Where are we today? | What does the global EV picture look like in one glance? |
| *1. Adoption Trends* | Who is winning? | Which countries grew fastest, and when did growth go exponential? |
| *2. Infrastructure & Demand* | Chicken or egg? | Do chargers lead EVs, or follow them? |
| *3. Market Composition* | What kind of EV? | BEV vs PHEV split, and is fleet turnover fast enough for 2030? |
| *4. Socioeconomic & Policy* | Is adoption equitable? | How does income shape EV uptake, and who are the outliers? |
| *5. Forecasting* | Where are we headed? | What does 2030 look like, and who hits the 30% threshold? |


In [10]:
# !pip install pandas numpy plotly statsmodels --quiet

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = "plotly_white"

# Brand palette
PALETTE = {
    "primary":   "#0B7A75",   # teal — historical / hero series
    "accent":    "#F4A261",   # amber — projection / highlight
    "bev":       "#264653",   # deep slate — BEV
    "phev":      "#E76F51",   # coral — PHEV
    "fcev":      "#9C89B8",   # lavender — FCEV (rare)
    "fast":      "#2A9D8F",   # green-teal — fast charger
    "slow":      "#A8DADC",   # pale teal — slow charger
    "muted":     "#94A3B8",   # gray — reference / context
    "danger":    "#D00000",
    "warning":   "#FFB703",
    "success":   "#06A77D",
}

REGION_COLORS = {
    "World": PALETTE["muted"],
    "China": "#D62828",
    "USA": "#1D4E89",
    "EU27": "#003566",
    "Europe": "#005F73",
    "India": "#F77F00",
    "Norway": "#0077B6",
    "Germany": "#FFB703",
    "France": "#7209B7",
    "United Kingdom": "#9D0208",
    "Japan": "#BC4749",
    "Korea": "#FF006E",
    "Brazil": "#3A5A40",
    "Australia": "#FB8500",
    "Canada": "#9B2226",
}

# Page-level CSS for HTML-exported charts
PAGE_FONT = "Arial, Helvetica, sans-serif"

pio.templates["dashboard"] = go.layout.Template(
    layout=go.Layout(
        font=dict(
            family=PAGE_FONT,
            size=12,
            color="#222222"
        ),
        title=dict(
            font=dict(
                family=PAGE_FONT,
                size=20,
                color="#111111"
            )
        ),
        legend=dict(
            font=dict(
                family=PAGE_FONT,
                size=12
            )
        ),

        hoverlabel=dict(
            font=dict(
                family=PAGE_FONT
            )
        )
    )
)

pio.templates.default = "dashboard"

print("Environment ready. Plotly:", pio.templates.default)

Environment ready. Plotly: dashboard


## Load raw data

Quick shape & schema check before we transform anything.

In [11]:
CSV_PATH = "D:\Data_Projects\data-visualization-shiny\data\cleaned_data\ev2025.csv"

df = pd.read_csv(CSV_PATH)
print(f"Rows: {len(df):,}\nColumns: {df.shape[1]}")
print("agg_group :", sorted(df["agg_group"].dropna().unique().tolist()))
print("powertrain:", sorted(df["powertrain"].dropna().unique().tolist()))
print("unit      :", sorted(df["unit"].dropna().unique().tolist()))
print("parameter :", sorted(df["parameter"].dropna().unique().tolist()))
print("years     :", df["year"].min(), "–", df["year"].max())
df.head()


Rows: 14,962
Columns: 10
agg_group : ['Continent', 'Country', 'EU27', 'World']
powertrain: ['BEV', 'EV', 'FCEV', 'Fast Charger', 'PHEV', 'Slow Charger']
unit      : ['GWh', 'Million barrels per day', 'Oil displacement (Mlge)', 'Vehicles', 'charging points', 'percent']
parameter : ['Battery demand', 'EV charging points', 'EV sales', 'EV sales share', 'EV stock', 'EV stock share', 'Electricity demand', 'Oil displacement (Mbd)', 'Oil displacement (Mlge)']
years     : 2010 – 2030


,region_country,category,parameter,mode,powertrain,year,unit,value,agg_group,is_projection
0,World,Projection-STEPS,EV stock,2 and 3 wheelers,BEV,2030,Vehicles,170000000.0,World,True
1,World,Projection-STEPS,EV stock,Cars,BEV,2030,Vehicles,150000000.0,World,True
2,China,Projection-STEPS,EV stock,2 and 3 wheelers,BEV,2030,Vehicles,91000000.0,Country,True
3,China,Projection-STEPS,EV stock,Cars,BEV,2030,Vehicles,82000000.0,Country,True
4,World,Projection-STEPS,EV stock,Cars,PHEV,2030,Vehicles,82000000.0,World,True


## Data characteristics

The semantic meaning of `powertrain` depends on the selected `parameter` and should not be interpreted uniformly across the dataset.

- For EV technology breakdown metrics (`EV stock`, `EV sales`, `Battery demand`, `Oil displacement`), `powertrain` represents EV drivetrain categories: `{BEV, PHEV, FCEV}`. `mode` identifies vehicle classes and `unit` is reported in vehicles.

- For aggregated EV indicators (`Electricity demand`, `EV stock share`, `EV sales share`), `powertrain` is fixed to `EV`, representing rolled-up EV totals across vehicle classes. Units are expressed in `GWh` or `%`.

- For charging infrastructure metrics (`EV charging point`), `powertrain` represents charger types: `{Fast Charger, Slow Charger}`. In this context, `mode` is fixed to `EV` and units are reported as charging points.

Important: always validate both `parameter` and `powertrain` before aggregation or comparison to avoid mixing incompatible concepts (e.g., vehicle counts, aggregated EV indicators, and charging infrastructure metrics).

## Helper functions

In [92]:
def slice_data(df, parameter=None, mode=None, powertrain=None,
               region=None, agg_group=None, category=None, year=None):
    """Filter data based on columns and returns a copy."""
    out = df
    for col, val in [("parameter", parameter), ("mode", mode), ("powertrain", powertrain),
                     ("region_country", region), ("agg_group", agg_group),
                     ("category", category), ("year", year)]:
        if val is None:
            continue
        if isinstance(val, (list, tuple, set)):
            out = out[out[col].isin(val)]
        else:
            out = out[out[col] == val]
    return out.copy()


def total_ev_stock_by(df, group_cols, modes=("Cars",)):
    """Sum all types of EV for given modes, group by columns and returns DataFrame"""
    sub = slice_data(df, parameter="EV stock",
                     mode=list(modes),
                     powertrain=["BEV","PHEV","FCEV"])
    return sub.groupby(list(group_cols), as_index=False)["value"].sum()


def total_chargers_by(df, group_cols):
    """Total two types of chargers and group by columns"""
    sub = slice_data(df, parameter="EV charging points",
                     powertrain=["Fast Charger","Slow Charger"])
    return sub.groupby(list(group_cols), as_index=False)["value"].sum()


# Check
demo = total_ev_stock_by(df, ["region_country","year"]).query("region_country == 'World'")
print("Total number of EV car on the road globally over time:")
print(demo.sort_values("year").to_string(index=False))

Total number of EV car on the road globally over time:
region_country  year       value
         World  2010     20426.0
         World  2011     67526.0
         World  2012    190026.0
         World  2013    390027.0
         World  2014    710087.0
         World  2015   1250810.0
         World  2016   2003100.0
         World  2017   3106800.0
         World  2018   5111000.0
         World  2019   7219000.0
         World  2020  10226000.0
         World  2021  16342000.0
         World  2022  26057000.0
         World  2023  40066000.0
         World  2024  58070000.0
         World  2030 232220000.0


In [94]:
def slice_data(df, parameter=None, mode=None, powertrain=None,
               region=None, agg_group=None, category=None, year=None):
    """Filter data based on columns and returns a copy."""
    out = df
    for col, val in [("parameter", parameter), ("mode", mode), ("powertrain", powertrain),
                     ("region_country", region), ("agg_group", agg_group),
                     ("category", category), ("year", year)]:
        if val is None:
            continue
        if isinstance(val, (list, tuple, set)):
            out = out[out[col].isin(val)]
        else:
            out = out[out[col] == val]
    return out.copy()


def total_ev_stock_by(df, group_cols, modes=("Cars",)):
    """Sum all types of EV for given modes, group by columns and returns DataFrame"""
    sub = slice_data(df, parameter="EV stock",
                     mode=list(modes),
                     powertrain=["BEV","PHEV","FCEV"])
    return sub.groupby(list(group_cols), as_index=False)["value"].sum()


def total_chargers_by(df, group_cols):
    """Total two types of chargers and group by columns"""
    sub = slice_data(df, parameter="EV charging points",
                     powertrain=["Fast Charger","Slow Charger"])
    return sub.groupby(list(group_cols), as_index=False)["value"].sum()


# Check
demo = total_ev_stock_by(df, ["region_country","year"]).query("region_country == 'World'")
print("Total number of EV car on the road globally over time:")
print(demo.sort_values("year").to_string(index=False))

Total number of EV car on the road globally over time:
region_country  year       value
         World  2010     20426.0
         World  2011     67526.0
         World  2012    190026.0
         World  2013    390027.0
         World  2014    710087.0
         World  2015   1250810.0
         World  2016   2003100.0
         World  2017   3106800.0
         World  2018   5111000.0
         World  2019   7219000.0
         World  2020  10226000.0
         World  2021  16342000.0
         World  2022  26057000.0
         World  2023  40066000.0
         World  2024  58070000.0
         World  2030 232220000.0


---

## 0. General Information

> Electric car sales topped 17 million worldwide in 2024, rising by more than 25%. Despite uncertainties in the outlook, the share of electric cars in overall car sales is set to exceed 40% in 2030 under today's policy settings. 

In [13]:
LATEST_HIST = int(df.query("category == 'Historical'")["year"].max())
HORIZON     = 2030

def kpi(label, value, sub=""):
    return dict(label=label, value=value, sub=sub)

# KPI 1: total EV cars on the road, world, latest year
stock_world_latest = total_ev_stock_by(df, ["region_country","year"]) \
    .query("region_country == 'World' and year == @LATEST_HIST")["value"].sum()

# KPI 2: EV stock share, world, latest year
stock_share_latest = slice_data(df, parameter="EV stock share",
                                 region="World", mode="Cars",
                                 powertrain="EV", year=LATEST_HIST)["value"].sum()

# KPI 3: EV sales share, world, latest year
sales_share_latest = slice_data(df, parameter="EV sales share",
                                 region="World", mode="Cars",
                                 powertrain="EV", year=LATEST_HIST)["value"].sum()

# KPI 4: total public chargers, world, latest year
chargers_world_latest = total_chargers_by(df, ["region_country","year"]) \
    .query("region_country == 'World' and year == @LATEST_HIST")["value"].sum()

# KPI 5: projected world EV car stock in 2030
stock_world_2030 = total_ev_stock_by(df, ["region_country","year"]) \
    .query("region_country == 'World' and year == @HORIZON")["value"].sum()

kpis = [
    kpi(f"EV cars on the road, {LATEST_HIST}",   f"{stock_world_latest/1e6:.1f}M"),
    kpi(f"Share of car stock, {LATEST_HIST}",     f"{stock_share_latest:.1f}%"),
    kpi(f"Share of new car sales, {LATEST_HIST}", f"{sales_share_latest:.1f}%"),
    kpi(f"Public chargers, {LATEST_HIST}",        f"{chargers_world_latest/1e6:.2f}M"),
]

# Render KPIs as a Plotly indicator strip
fig_kpi = make_subplots(rows=1, cols=4, specs=[[{"type":"indicator"}]*4])
for i, k in enumerate(kpis, start=1):
    fig_kpi.add_trace(go.Indicator(
        mode="number",
        value=float(k["value"].replace("M","").replace("%","")),
        number={"suffix": "M" if "M" in k["value"] else ("%" if "%" in k["value"] else ""),
                "font": {"size": 36, "color": PALETTE["primary"]}},
        title={"text": f"<span style='font-size:12px;color:#475569'>{k['label']}</span>"},
    ), row=1, col=i)

# Edit center positions (optional)
# fig_kpi.update_layout(height=180, margin=dict(l=5, r=200, t=20, b=5),
#                       paper_bgcolor="white",
#                       title=dict(text=f"<b>Global EV figures in 2024</b>",
#                       x=0, y=0.9, font=dict(size=16)))

fig_kpi.show()

In [14]:
list(np.sort(df["year"].unique()))

[np.int64(2010),
 np.int64(2011),
 np.int64(2012),
 np.int64(2013),
 np.int64(2014),
 np.int64(2015),
 np.int64(2016),
 np.int64(2017),
 np.int64(2018),
 np.int64(2019),
 np.int64(2020),
 np.int64(2021),
 np.int64(2022),
 np.int64(2023),
 np.int64(2024),
 np.int64(2030)]

In [15]:
sales_share = df[(df["parameter"] == "EV sales share") & (df["mode"] == "Cars") & (df["region_country"] == "World")]

# Separate historical and projection
sales_hist = sales_share[sales_share["category"] == "Historical"].sort_values("year")
sales_proj = sales_share[sales_share["category"] == "Projection-STEPS"].sort_values("year")

fig = go.Figure()

# Add historical line
if len(sales_hist) > 0:
    fig.add_trace(go.Scatter(
        x=sales_hist["year"], y=sales_hist["value"],
        mode="lines+markers", name="Historical",
        line=dict(color=PALETTE["primary"], width=2.5),
        marker=dict(size=8),
        hovertemplate="%{x}: %{y:.1f}%<extra></extra>"
    ))

# Add projection line (dashed) from 2024 to 2030
if len(sales_proj) > 0:
    # Create a bridge from last historical to first projection
    bridge_x = [sales_hist["year"].iloc[-1]] + sales_proj["year"].tolist()
    bridge_y = [sales_hist["value"].iloc[-1]] + sales_proj["value"].tolist()
    
    fig.add_trace(go.Scatter(
        x=bridge_x, y=bridge_y,
        mode="lines+markers", name="Projection (STEPS)",
        line=dict(color=PALETTE["accent"], width=2.5, dash="dash"),
        marker=dict(size=8),
        hovertemplate="%{x}: %{y:.1f}%<extra></extra>",
    ))
    
    # Add annotation for 2030 point
    fig.add_annotation(
        x=2030, y=bridge_y[-1],
        text="Expected to exceed 40%",
        showarrow=True,
        arrowhead=2,
        ax=0, ay=-40,
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor=PALETTE["accent"],
        borderpad=4,
        font=dict(size=11, color="#111111")
    )

fig.update_layout(
    height=500,
    xaxis_title="Year",
    yaxis_title="EV sales share (%)",
    title="<b>EV sales share of new cars (World)</b>",
    hovermode="x unified"
)

fig.show()

In [16]:
nv_stock = df[(df["parameter"]=="EV stock") & (df["year"]==LATEST_HIST) & (df["agg_group"]=="Country")]
heat = nv_stock.groupby(["mode","powertrain"])["value"].median().reset_index()
heat_piv = heat.pivot(index="mode", columns="powertrain", values="value").fillna(0)
heat_piv = heat_piv.reindex(columns=["BEV","PHEV","FCEV"])

fig1c = go.Figure(go.Heatmap(
    z=np.log10(heat_piv.values),
    x=heat_piv.columns, y=heat_piv.index,
    text=[[f"{v:,.0f}" for v in row] for row in heat_piv.values],
    texttemplate="%{text}", textfont=dict(size=12),
    colorscale="tempo",
    hovertemplate="<b>%{y}</b> × %{x}<br>Median: %{text} vehicles<extra></extra>",
    colorbar=dict(title="log10(vehicles)", thickness=12)
))

fig1c.update_layout(height=400, margin=dict(t=60,b=60,l=50,r=500),
                    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top",
                               text=f"<b>EV stocks in average by mode and powertrain, 2024</b>"))
fig1c.show()

In [17]:
nv_stock

,region_country,category,parameter,mode,powertrain,year,unit,value,agg_group,is_projection
9,China,Historical,EV stock,2 and 3 wheelers,BEV,2024,Vehicles,67000000.0,Country,False
32,China,Historical,EV stock,Cars,BEV,2024,Vehicles,23000000.0,Country,False
58,China,Historical,EV stock,Cars,PHEV,2024,Vehicles,11000000.0,Country,False
95,India,Historical,EV stock,2 and 3 wheelers,BEV,2024,Vehicles,6600000.0,Country,False
122,USA,Historical,EV stock,Cars,BEV,2024,Vehicles,4700000.0,Country,False
...,...,...,...,...,...,...,...,...,...,...
11199,Norway,Historical,EV stock,Trucks,PHEV,2024,Vehicles,1.0,Country,False
11211,Portugal,Historical,EV stock,Buses,FCEV,2024,Vehicles,1.0,Country,False
11215,Rest of the world,Historical,EV stock,Trucks,FCEV,2024,Vehicles,1.0,Country,False
11230,Sweden,Historical,EV stock,Trucks,FCEV,2024,Vehicles,1.0,Country,False


In [70]:
nv_stock = df[(df["parameter"]=="EV stock") & (df["year"]==LATEST_HIST) & (df["agg_group"]=="Country")]
nv_stock["mode"] = nv_stock["mode"].replace({
    "Trucks": "Other (Trucks, Buses, Vans)",
    "Buses": "Other (Trucks, Buses, Vans)",
    "Vans": "Other (Trucks, Buses, Vans)"
})

# Group by mode and powertrain
data = nv_stock.groupby(["mode","powertrain"], as_index=False)["value"].sum()

# Create subplots with 3 columns (one for each powertrain)
fig1c = make_subplots(
    rows=1, cols=3,
    subplot_titles=("BEV", "PHEV", "FCEV"),
    specs=[[{"type":"pie"}, {"type":"pie"}, {"type":"pie"}]]
)

powertrains = ["BEV", "PHEV", "FCEV"]
colors_mode = px.colors.qualitative.Set3

for col_idx, pt in enumerate(powertrains, start=1):
    # Filter for this powertrain
    pt_data = data[data["powertrain"] == pt].sort_values("value", ascending=False)
    
    fig1c.add_trace(
        go.Pie(
            labels=pt_data["mode"],
            values=pt_data["value"],
            name=pt,
            marker=dict(colors=colors_mode),
            hovertemplate="<b>%{label}</b><br>%{value:,.0f} vehicles<br>%{percent}<extra></extra>",
            rotation=-45
        ),
        row=1, col=col_idx
    )

fig1c.update_layout(
    height=450,
    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top",
               text=f"<b>EV stock proportion by mode and powertrain, {LATEST_HIST}</b>"),
    # margin=dict(t=60,b=60,l=50,r=500),
    showlegend=True
)

fig1c.show()

In [21]:
# ---- Global trajectory: stock + sales share ----

# stock series, historical + 2030 projection
stock_series = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World'")
stock_hist = stock_series[stock_series["category"]=="Historical"].sort_values("year")
stock_proj = stock_series[stock_series["category"]=="Projection-STEPS"].sort_values("year")

# sales-share series
share = slice_data(df, parameter="EV sales share",
                    region="World", mode="Cars", powertrain="EV")
share_hist = share[share["category"]=="Historical"].sort_values("year")
share_proj = share[share["category"]=="Projection-STEPS"].sort_values("year")

# --- Determine stock y-axis max ---
import math
stock_max = max(
    stock_hist["value"].max()/1e6,
    stock_proj["value"].max()/1e6 if len(stock_proj) else 0
)
stock_y_max = math.ceil(stock_max / 50) * 50  # round up to nearest 50


fig0 = make_subplots(specs=[[{"secondary_y": True}]])
# --- Stock as BAR CHART ---
# Historical bars
fig0.add_trace(go.Bar(
    x=stock_hist["year"], y=stock_hist["value"]/1e6,
    name="EV car stock (2010-2024)",
    marker_color=PALETTE["primary"],
    opacity=0.75,
    hovertemplate="%{y:.1f}M EV cars<extra></extra>",
), secondary_y=False)

# Projected bars (2030)
fig0.add_trace(go.Bar(
    x=stock_proj["year"], y=stock_proj["value"]/1e6,
    name="EV car stock (2030 proj.)",
    marker_color=PALETTE["primary"],
    opacity=0.4,
    marker_line=dict(color=PALETTE["primary"], width=1.5),
    hovertemplate="%{y:.1f}M EV cars (projected)<extra></extra>",
), secondary_y=False)

# --- Sales share as SCATTER PLOT ---
# Historical scatter
fig0.add_trace(go.Scatter(
    x=share_hist["year"], y=share_hist["value"],
    mode="markers", name="EV sales share (2010-2024)",
    marker=dict(color=PALETTE["warning"], size=10, symbol="circle",
                line=dict(color="white", width=1.5)),
    hovertemplate="%{y:.1f}% of new cars<extra></extra>",
), secondary_y=True)

# Projected scatter (2030)
if len(share_proj):
    fig0.add_trace(go.Scatter(
        x=share_proj["year"], y=share_proj["value"],
        mode="markers", name="EV sales share (2030 proj.)",
        marker=dict(color=PALETTE["warning"], size=12, symbol="star",
                    line=dict(color="white", width=1.5), opacity=0.7),
        hovertemplate="%{y:.1f}% of new cars (projected)<extra></extra>",
    ), secondary_y=True)

# --- Y-axes: aligned at 0, independent scales ---
fig0.update_yaxes(
    title_text="EV cars on the road (millions)",
    range=[0, stock_y_max],
    secondary_y=False,
)
fig0.update_yaxes(
    title_text="Share of new car sales (%)",
    range=[0, 80],
    dtick=20,  # 0, 20, 40, 60, 80, 100
    secondary_y=True,
)

# --- X-axis: show every year ---
# --- Collect all years for x-axis ---
all_years = sorted(
    stock_hist["year"].tolist() +
    stock_proj["year"].tolist()
)

fig0.update_xaxes(
    title_text="Year",
    tickmode="array",
    tickvals=all_years,
    tickangle=-45,
)

fig0.update_layout(
    barmode="stack",
    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top",
               text="<b>Current and projected trends of EV Car Fleet & Sales Share</b>"),
    legend=dict(title=None, orientation="h", x=0.4, y=1.2, xanchor="right", yanchor="top"),
)

fig0.show()

In [77]:
nv_world = df[(df["agg_group"]=="World") & (df["parameter"]=="EV stock") & (df["mode"]=="Cars")]
nv_world_pt = nv_world.groupby(["year","powertrain"], as_index=False)["value"].sum()

fig1e = px.area(
    nv_world_pt.sort_values("year"), x="year", y="value", color="powertrain",
    category_orders={"powertrain": ["FCEV", "PHEV", "BEV"]},
    labels={"value":"EV stock (vehicles)","year":"Year","powertrain":"Powertrain"},
    title="<b>World EV car stocks by powertrain over time</b>",
)

fig1e.update_traces(hovertemplate="<b>%{fullData.name}</b><br>Year: %{x}<br>EV Stock: %{y:,.0f}<extra></extra>")
fig1e.update_layout(title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
                    legend=dict(title=None, orientation="h", x=0.5, y=1.05, xanchor="left", yanchor="top"))
fig1e.show()

In [23]:
ch_world = df[(df["agg_group"]=="World") & (df["unit"]=="charging points")].groupby(["year","powertrain"], as_index=False)["value"].sum()

fig3a = px.bar(
    ch_world.sort_values("year"), x="year", y="value", color="powertrain", barmode="stack",
    labels={"value":"Charging points","year":"Year","powertrain":"Charger type"},
    title="<b>World charging points over time</b>",
)

# --- X-axis: show every year ---
# --- Collect all years for x-axis ---
all_years = sorted(ch_world["year"].tolist())

fig3a.update_xaxes(
    title_text="Year",
    tickmode="array",
    tickvals=all_years,
    tickangle=-45,
)

fig3a.update_layout(
                    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
                    legend=dict(title=None, orientation="h", x=0.5, y=1.05, xanchor="left", yanchor="top"))
fig3a.show()

---

## TAB 1: Adoption Trends
1. **Which countries saw the fastest EV adoption growth from 2010 to 2024?**

* </p>China started to race after US on EV production game from 2012 and officialy take over the dominance from 2016. Overall, there are 3 big players in the global market: China, USA, EU.

*2. What factors drive EV growth: charging infrastructure or other socioeconomic conditions?*


*3. When did major EV markets transition from linear to exponential sales growth?*

In [95]:
# ---- Load iso_code from merged_output and merge with ev data ----
merged_output = pd.read_csv("D:\\Data_Projects\\data-visualization-shiny\\data\\merged_output.csv")
iso_map = merged_output[["region_country", "code"]].drop_duplicates().rename(columns={"code":"iso_code"})

stock = df[(df["parameter"]=="EV stock") & (df["year"]==LATEST_HIST) & (df["agg_group"] != "World")]
sales = df[(df["parameter"]=="EV sales") & (df["year"]==LATEST_HIST) & (df["agg_group"] != "World")]

# Group by country only (not by mode)
stock_agg = stock.groupby(["region_country"], as_index=False)["value"].sum().rename(columns={"value":"stock"})
sales_agg = sales.groupby(["region_country"], as_index=False)["value"].sum().rename(columns={"value":"sales"})

cluster = stock_agg.merge(sales_agg, on=["region_country"], how="outer").fillna(0)
cluster = cluster.merge(iso_map, on="region_country", how="left")
cluster = cluster[(cluster["stock"]>0) & (cluster["sales"]>0)].copy()

# # Unit (M)
cluster["stock"] = cluster["stock"] / 1e6
cluster["sales"] = cluster["sales"] / 1e6

cluster["sales_to_stock"] = np.round(cluster["sales"] / cluster["stock"], 2)

fig1d = px.scatter(
    cluster, x="stock", y="sales", color="iso_code",
    hover_name="region_country",
    size="stock", size_max=100,
    labels={"stock":"EV stock (Million)","sales":"EV sales (Million)","iso_code":"Country Code"},
    title=f"<b>EV sales and actual on the road by country, {LATEST_HIST}</b>"
)

# ---- Labels to annotate ----
highlight = ["China", "Asia Pacific", "Europe", "USA", "North America", "India"]
annot_df = cluster[cluster["region_country"].isin(highlight)]

print(f"Top players: {annot_df["region_country"].values}")

# Custom annotation positions
annotation_positions = {
    "China": dict(ax=0, ay=0),
    "Asia Pacific": dict(ax=0, ay=0),
    "Europe": dict(ax=0, ay=0),
    "USA": dict(ax=0, ay=0), # keep USA centered
    "North America": dict(ax=35, ay=5),
    "India": dict(ax=0, ay=25), # move India above bubble
}

# Add annotations
for _, row in annot_df.iterrows():
    pos = annotation_positions.get(
        row["region_country"], dict(ax=0, ay=0)
    )

    fig1d.add_annotation(
        x=row["stock"],  y=row["sales"],
        text=row["iso_code"], showarrow=False,
        xshift=pos["ax"], yshift=pos["ay"],
        font=dict(size=12, color="black")
    )

fig1d.update_layout(
                    # height=600, margin=dict(t=60,b=60,l=0,r=500),
                    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
                    legend=dict(title="Country", orientation="v", x=1.02, y=1, xanchor="left", yanchor="top"))
fig1d.show()

Top players: ['Asia Pacific' 'China' 'Europe' 'India' 'North America' 'USA']


### Chart 1.2: Race bar: top-15 country EV stock

The race bar show accumulated volume of EV stock of countries throughout the year. It starts with USA being in the first place then Japan also an outstanding player, but then USA climbs steadily and wins the highest with latter are European notable countries.

In [116]:
stock_country = total_ev_stock_by(df, ["region_country", "year", "category"])
stock_country = stock_country[(stock_country["category"]=="Historical") &
                              (stock_country["year"] >= 2010)]
stock_country = stock_country.merge(
    df[["region_country","agg_group"]].drop_duplicates(),
    on="region_country", how="left"
)
stock_country = stock_country[stock_country["agg_group"] == "Country"].copy()

# For each year keep top 15
frames = []
for y in sorted(stock_country["year"].unique()):
    sub = stock_country[stock_country["year"] == y].nlargest(15, "value").sort_values("value")
    sub = sub.assign(year=y)
    frames.append(sub)
race_df = pd.concat(frames, ignore_index=True)
race_df["value_M"] = race_df["value"] / 1e6

def race_color(c):
    return REGION_COLORS.get(c, PALETTE["primary"])

years_race = sorted(race_df["year"].unique())

# ---- Use the LATEST year's top-15 as the fixed initial country list ----
latest_countries = race_df[race_df["year"] == years_race[-1]].sort_values("value")["region_country"].tolist()

# ---- Dynamic x-axis scaling function ----
def compute_xrange(max_val):
    """Scale x-axis so top bar never exceeds ~60% of visible range."""
    if max_val <= 0:
        return [0, 0.1]  # small default for the zero-frame
    # top bar should sit at 60% of axis → axis max = max_val / 0.6
    axis_max = max_val / 0.6
    return [0, axis_max]

# ---- Initial frame: all zeros, using latest top-15 countries ----
fig_race = go.Figure()
fig_race.add_trace(go.Bar(
    x=[0] * len(latest_countries),
    y=latest_countries,
    orientation="h",
    marker=dict(color=[race_color(c) for c in latest_countries]),
    text=["0.00 M"] * len(latest_countries),
    textposition="outside",
    hovertemplate="<b>%{y}</b>: %{x:.2f} M EVs<extra></extra>",
))

# ---- Build animated frames ----
fig_frames = []
for y in years_race:
    sub = race_df[race_df["year"] == y]
    # Reindex to the top-15 of THIS year, sorted by value
    sub_sorted = sub.sort_values("value")
    countries = sub_sorted["region_country"].tolist()
    values = sub_sorted["value_M"].tolist()
    
    max_val = max(values) if values else 0
    xrange = compute_xrange(max_val)
    
    fig_frames.append(go.Frame(
        name=str(y),
        data=[go.Bar(
            x=values, y=countries, orientation="h",
            marker=dict(color=[race_color(c) for c in countries]),
            text=[f"{v:.2f} M" for v in values],
            textposition="outside",
        )],
        layout=go.Layout(
            title_text=f"<b>Top-15 EV car stock by country — {y}</b>",
            xaxis=dict(range=xrange),
        )
    ))
fig_race.frames = fig_frames

fig_race.update_layout(
    title=f"<b>Top-15 EV car stock by country — Start</b>",
    height=560,
    xaxis_title="EV cars on the road (millions)",
    xaxis=dict(range=[0, 0.1]),  # tight range for initial zero state
    yaxis_title="",
    margin=dict(l=140, r=80, t=70, b=80),
    updatemenus=[{
        "type": "buttons", "showactive": False, "y": -0.2, "x": 0,
        "direction": "right",
        "buttons": [
            {"label": "▶", "method": "animate",
             "args": [None, {"frame": {"duration": 1500, "redraw": True}, "fromcurrent": True}]},
            {"label": "❚❚", "method": "animate",
             "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]},
        ],
    }],
    sliders=[{
        "active": 0, "y": -0.10, "x": 0.05, "len": 0.92,
        "currentvalue": {"prefix": "Year: ", "font": {"size": 14, "color": PALETTE["primary"]}},
        "steps": [{"label": str(y), "method": "animate",
                   "args": [[str(y)], {"frame": {"duration": 500, "redraw": True},
                                       "mode": "immediate"}]} for y in years_race],
    }],
)

fig_race.show()

### Chart 1.2: EV Growth Comparison of Top 3 in recent years

In [115]:
# ---- Prepare data ----
regions = ["China", "USA", "Europe"]
growth_data = df[
    (df["mode"] == "Cars") &
    (df["category"] == "Historical") &
    (df["year"] >= 2016) &
    (df["year"] <= 2024) &
    (df["region_country"].isin(regions))
].copy()

stock_by_pt = growth_data[growth_data["parameter"] == "EV stock"].copy()
stock_by_pt = stock_by_pt.groupby(["region_country", "year", "powertrain"], as_index=False)["value"].sum()

share_data = growth_data[growth_data["parameter"] == "EV sales share"].copy()
share_data = share_data.groupby(["region_country", "year"], as_index=False)["value"].sum()

stock_pivot = stock_by_pt.pivot_table(
    index=["region_country", "year"], columns="powertrain", values="value", fill_value=0
).reset_index()
for pt in ["BEV", "PHEV", "FCEV"]:
    if pt not in stock_pivot.columns:
        stock_pivot[pt] = 0

# ---- Compute global max for both axes ----
max_stock = stock_pivot[["BEV", "PHEV", "FCEV"]].sum(axis=1).max()
max_share = share_data["value"].max()

# Add some headroom
stock_range = [0, max_stock * 1.1]
share_range = [0, max_share * 1.15]

# Nice share ticks with % suffix
import numpy as np
share_tick_step = 10 if max_share > 30 else 5
share_tickvals = list(np.arange(0, share_range[1] + 1, share_tick_step))
share_ticktext = [f"{int(v)}%" for v in share_tickvals]

# ---- Create subplots ----
fig_growth = make_subplots(
    rows=1, cols=3,
    specs=[[{"secondary_y": True}, {"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=["China", "USA", "Europe"],
    horizontal_spacing=0.06
)

regions_list = ["China", "USA", "Europe"]
colors_pt = {"BEV": PALETTE["fast"], "PHEV": PALETTE["slow"], "FCEV": PALETTE["success"]}
share_color = PALETTE["accent"]

for col_idx, region in enumerate(regions_list, 1):
    region_stock = stock_pivot[stock_pivot["region_country"] == region].sort_values("year")
    region_share = share_data[share_data["region_country"] == region].sort_values("year")

    for pt in ["BEV", "PHEV", "FCEV"]:
        fig_growth.add_trace(
            go.Bar(
                x=region_stock["year"], y=region_stock[pt],
                name=pt, marker_color=colors_pt[pt],
                legendgroup=pt, showlegend=(col_idx == 1),
                hovertemplate=f"<b>{pt}</b><br>Year: %{{x}}<br>Stock: %{{y:,.0f}}<extra></extra>",
            ),
            row=1, col=col_idx, secondary_y=False
        )

    fig_growth.add_trace(
        go.Scatter(
            x=region_share["year"], y=region_share["value"],
            mode="markers+lines", name="EV Share",
            marker=dict(size=8, color=share_color),
            line=dict(color=share_color, width=2, dash="dash"),
            legendgroup="share", showlegend=(col_idx == 1),
            hovertemplate="<b>EV Share</b><br>Year: %{x}<br>Share: %{y:.1f}%<extra></extra>",
        ),
        row=1, col=col_idx, secondary_y=True
    )

# ---- Unify all y-axes ----
for col_idx in range(1, 4):
    # Left y-axis (stock): same range, only show title & ticks on col 1
    fig_growth.update_yaxes(
        range=stock_range, row=1, col=col_idx, secondary_y=False,
        showticklabels=(col_idx == 1),
        title_text="EV Stock (vehicles)" if col_idx == 1 else "",
    )
    # Right y-axis (share): same range, only show title & ticks on col 3
    fig_growth.update_yaxes(
        range=share_range, row=1, col=col_idx, secondary_y=True,
        showticklabels=(col_idx == 3),
        title_text="EV Share (%)" if col_idx == 3 else "",
        tickvals=share_tickvals if col_idx == 3 else [],
        ticktext=share_ticktext if col_idx == 3 else [],
    )

fig_growth.update_layout(
    title="<b>EV Growth Comparison: China, USA, Europe (2016-2024)</b>",
    height=500, barmode="stack", hovermode="x unified",
    legend=dict(orientation="h", x=0.35, y=1.15),
    margin=dict(l=80, r=80, t=100, b=80),
)

fig_growth.show()

### Chart 1.4 (optional): China vs United States with some events

In [26]:
sales = slice_data(df, parameter="EV sales",
                   mode="Cars", powertrain=["BEV","PHEV","FCEV"],
                   region=["China", "USA"],
                   category="Historical")
sales = sales.groupby(["region_country","year"], as_index=False)["value"].sum()

# policy event markers
policy_events = [
  {"year": 2012, "country": "USA",   "label": "Tesla Model S released"},
  {"year": 2017, "country": "China", "label": "NEV credit mandate announced"},
  {"year": 2020, "country": "China", "label": "Mass charging infrastructure rollout"},
  {"year": 2021, "country": "USA",   "label": "Biden EV target announced"},
  {"year": 2023, "country": "China", "label": "BYD surpasses Tesla sales"},
  {"year": 2023, "country": "USA",   "label": "Tesla charging standard adopted"},
]

fig_infl = go.Figure()
for country in ["China", "USA"]:
    sub = sales[sales["region_country"] == country].sort_values("year")
    fig_infl.add_trace(go.Scatter(
        x=sub["year"], y=sub["value"],
        mode="lines+markers", name=country,
        line=dict(color=REGION_COLORS[country], width=3),
        marker=dict(size=8),
        hovertemplate=f"<b>{country}</b><br>%{{x}}: %{{y:,.0f}} EVs sold<extra></extra>",
    ))

# overlay policy events
for ev in policy_events:
    sub = sales[(sales["region_country"]==ev["country"]) & (sales["year"]==ev["year"])]
    if len(sub) == 0: continue
    y_val = sub["value"].iloc[0]
    fig_infl.add_annotation(
        x=ev["year"], y=y_val,
        text=f"<b>{ev['country'][:2]}</b> · {ev['label']}",
        showarrow=True, arrowhead=2, arrowcolor=PALETTE["muted"],
        ax=0, ay=-50, bgcolor="rgba(255,255,255,0.92)",
        bordercolor=PALETTE["muted"], borderpad=4,
        font=dict(size=10, color="#1f2937"),
    )

fig_infl.update_yaxes(type="linear", title_text="EV sales (vehicles, log scale)")
fig_infl.update_xaxes(title_text="Year")
fig_infl.update_layout(
    title=dict(x=0.02, y=0.98, xanchor="left", yanchor="top",
               text="<b>When did exponential growth start?</b>"),
    height=500, hovermode="x unified",
    legend=dict(orientation="h", y=1, x=0.4, xanchor="left", yanchor="top"),
    margin=dict(t=60,b=60,l=50,r=300),
)

# fig3a.update_layout(height=400, margin=dict(t=60,b=60,l=0,r=500),
#                     title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
#                     legend=dict(title=None, orientation="h", x=0.5, y=1.05, xanchor="left", yanchor="top"))
fig_infl.show()


---

## TAB 2: Infrastructure & Demand
(a) Is charging infrastructure a leading or lagging indicator of EV adoption?

(b) Which countries have the lowest charger-to-EV availability?


### Chart 2.1: total EVs vs total public chargers in the world

If chargers lead EVs, the charger curve should be steeper or earlier. If they lag, EVs lead.

In [27]:
# ---- Dual area chart: stock vs chargers, world ----

# Two area subplots stacked vertically so reader can compare *shapes* not magnitudes
ev_world = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World' and category == 'Historical'") \
    .sort_values("year")
ch_world = total_chargers_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World' and category == 'Historical'") \
    .sort_values("year")

# also break chargers into fast vs slow
ch_split = slice_data(df, parameter="EV charging points",
                       region="World", category="Historical",
                       powertrain=["Fast Charger","Slow Charger"])
ch_split = ch_split.groupby(["year","powertrain"], as_index=False)["value"].sum()

fig_inf = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("EV cars on the road (BEV+PHEV+FCEV, all modes summed = Cars)",
                                        "Public chargers (Fast vs Slow)"),
                        vertical_spacing=0.12)

fig_inf.add_trace(go.Scatter(
    x=ev_world["year"], y=ev_world["value"]/1e6,
    fill="tozeroy", mode="lines", name="EV stock (M)",
    line=dict(color=PALETTE["primary"], width=2.5),
    fillcolor="rgba(11,122,117,0.22)",
    hovertemplate="%{x}: %{y:.2f} M EVs<extra></extra>",
), row=1, col=1)

for pt, color in [("Slow Charger", PALETTE["slow"]), ("Fast Charger", PALETTE["fast"])]:
    sub = ch_split[ch_split["powertrain"]==pt].sort_values("year")
    fig_inf.add_trace(go.Scatter(
        x=sub["year"], y=sub["value"]/1e6,
        mode="lines", name=pt,
        stackgroup="chargers",
        line=dict(color=color, width=1.5),
        fillcolor=color,
        hovertemplate=f"<b>{pt}</b> %{{x}}: %{{y:.2f}} M points<extra></extra>",
    ), row=2, col=1)

fig_inf.update_xaxes(title_text="Year", row=2, col=1)
fig_inf.update_yaxes(title_text="EVs (millions)", row=1, col=1)
fig_inf.update_yaxes(title_text="Charging points (millions)", row=2, col=1)
fig_inf.update_layout(
    title="<b>Chargers vs EVs: do they grow together?</b> · world, historical",
    height=620, hovermode="x unified",
    legend=dict(orientation="h", y=-0.10, x=0.5, xanchor="center"),
    margin=dict(l=60,r=60,t=80,b=70),
)
fig_inf.show()


### Chart 2.2: Charger-to-EV ratio: which countries are most under-built?

New Zealand, Australia, Mexico, USA, Norway

In [126]:
latest = LATEST_HIST

ev_country = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("category == 'Historical' and year == @latest")

ch_country = total_chargers_by(df, ["region_country","year","category"]) \
    .query("category == 'Historical' and year == @latest")

ratio = ev_country.merge(
    ch_country,
    on=["region_country","year","category"],
    suffixes=["_ev","_ch"]
)

# filter to countries only
ratio = ratio.merge(
    df[["region_country","agg_group"]].drop_duplicates(),
    on="region_country"
)

ratio = ratio[
    (ratio["agg_group"] == "Country") &
    (ratio["region_country"] != "Rest of the world") &
    (ratio["value_ev"] >= 5_000) &
    (ratio["value_ch"] > 0)
].copy()

ratio["evs_per_charger"] = ratio["value_ev"] / ratio["value_ch"]
ratio["ev_stock_M"]      = ratio["value_ev"] / 1e6

# ---------------------------------------------------------
# continent lookup
# ---------------------------------------------------------
CONTINENT = {
    "China":"Asia","India":"Asia","Indonesia":"Asia","Japan":"Asia","Korea":"Asia",
    "Thailand":"Asia","Israel":"Asia","Turkey":"Asia","Turkiye":"Asia",

    "Germany":"Europe","France":"Europe","United Kingdom":"Europe","Norway":"Europe",
    "Sweden":"Europe","Netherlands":"Europe","Italy":"Europe","Spain":"Europe",
    "Belgium":"Europe","Switzerland":"Europe","Denmark":"Europe","Finland":"Europe",
    "Poland":"Europe","Austria":"Europe","Portugal":"Europe","Ireland":"Europe",
    "Greece":"Europe","Czech Republic":"Europe","Iceland":"Europe","Hungary":"Europe",
    "Slovakia":"Europe","Slovenia":"Europe","Romania":"Europe","Bulgaria":"Europe",
    "Croatia":"Europe","Estonia":"Europe","Latvia":"Europe","Lithuania":"Europe",
    "Luxembourg":"Europe","Malta":"Europe","Cyprus":"Europe","Ukraine":"Europe",

    "United States":"America","Canada":"America","Mexico":"America",
    "Brazil":"America","Chile":"America","Colombia":"America","Costa Rica":"America",

    "South Africa":"Africa",

    "Australia":"Oceania","New Zealand":"Oceania",
}

ratio["continent"] = ratio["region_country"].map(CONTINENT)
ratio["continent"] = ratio["continent"].fillna("Other")

CONTINENT_COLORS = {
    "Asia":"#D62828",
    "Europe":"#1D4E89",
    "America":"#9B2226",
    "Africa":"#FB8500",
    "Oceania":"#7209B7",
    "Other":"#94A3B8",
}

# ---------------------------------------------------------
# marker size scaling
# ---------------------------------------------------------
ratio["bubble_size"] = np.sqrt(ratio["value_ev"]) / 120

# ---------------------------------------------------------
# labels inside bubbles for large circles
# ---------------------------------------------------------
ratio["text_inside"] = np.where(
    ratio["bubble_size"] >= 12,
    ratio["region_country"],
    ""
)

# ---------------------------------------------------------
# scatter
# ---------------------------------------------------------
fig_ratio = px.scatter(
    ratio.sort_values("evs_per_charger"),

    x="ev_stock_M",
    y="evs_per_charger",

    size="bubble_size",
    size_max=60,

    color="continent",
    color_discrete_map=CONTINENT_COLORS,

    hover_name="region_country",

    # IMPORTANT
    text="text_inside",

    labels={
        "ev_stock_M":"EV stock (M, log scale)",
        "evs_per_charger":"EVs per public charger",
        "continent":"Continent"
    },

    title=f"<b>Charger-to-EV stress test, {latest}</b>",
)

# ---------------------------------------------------------
# bubble styling
# ---------------------------------------------------------
fig_ratio.update_traces(
    marker=dict(
        line=dict(width=1.2, color="white"),
        opacity=0.82
    ),

    # center text inside bubble
    textposition="middle center",

    textfont=dict(
        size=10,
        color="white"
    )
)

fig_ratio.show()

In [127]:
worst = ratio.nlargest(5, "evs_per_charger")

print("\nTop-5 worst charger-to-EV ratios:\n")

print(
    worst[
        ["region_country","ev_stock_M","value_ch","evs_per_charger"]
    ]
    .rename(columns={"value_ch":"chargers"})
    .to_string(index=False)
)


Top-5 worst charger-to-EV ratios:

region_country  ev_stock_M  chargers  evs_per_charger
   New Zealand    0.113038    1440.0        78.498611
     Australia    0.302089    6700.0        45.087910
        Mexico    0.068000    1850.0        36.756757
           USA    6.319000  193000.0        32.740933
        Norway    0.960230   31000.0        30.975161


---

## TAB 3: Market Composition
How does the BEV–PHEV market share differ across income groups?

### Chart 3.1: Stacked bar: BEV vs PHEV share by country, latest year

We rank the top 20 countries by total EV stock, then show the BEV/PHEV mix as a 100% stacked bar.

In [30]:
# ---- Stacked bar: BEV vs PHEV mix per country, latest year ----

mix = slice_data(df, parameter="EV stock", mode="Cars",
                  powertrain=["BEV","PHEV"],
                  category="Historical", year=LATEST_HIST,
                  agg_group="Country")
mix_p = mix.pivot_table(index="region_country", columns="powertrain",
                        values="value", aggfunc="sum", fill_value=0)
mix_p["total"] = mix_p.sum(axis=1)
mix_p = mix_p.nlargest(20, "total").sort_values("total")
mix_p["BEV_pct"]  = mix_p["BEV"]  / mix_p["total"] * 100
mix_p["PHEV_pct"] = mix_p["PHEV"] / mix_p["total"] * 100

fig_mix = go.Figure()
fig_mix.add_trace(go.Bar(
    y=mix_p.index, x=mix_p["BEV_pct"], orientation="h", name="BEV",
    marker_color=PALETTE["bev"],
    text=[f"{v:.0f}%" for v in mix_p["BEV_pct"]], textposition="inside",
    hovertemplate="<b>%{y}</b><br>BEV: %{x:.1f}%<extra></extra>",
))
fig_mix.add_trace(go.Bar(
    y=mix_p.index, x=mix_p["PHEV_pct"], orientation="h", name="PHEV",
    marker_color=PALETTE["phev"],
    text=[f"{v:.0f}%" for v in mix_p["PHEV_pct"]], textposition="inside",
    hovertemplate="<b>%{y}</b><br>PHEV: %{x:.1f}%<extra></extra>",
))
fig_mix.update_layout(barmode="stack", height=600,
                      title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top",
                                 text=f"<b>Top 20 EV markets proportion, {LATEST_HIST}</b>"),
                      xaxis_title="Share of EV car stock (%)",
                      margin=dict(t=60,b=60,l=0,r=500),
                      legend=dict(title=None, orientation="h", x=0.5, y=1.07, xanchor="left", yanchor="top"))

# fig1d.update_layout(height=600, margin=dict(t=60,b=60,l=0,r=500),
#                     title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
#                     legend=dict(title=None, orientation="h", x=0.4, y=1, xanchor="left", yanchor="top"))
fig_mix.show()


### Chart 3.2: Fleet turnover: sales share vs stock share

If sales share rises *much faster* than stock share, the fleet is turning over fast enough to hit 2030 targets. If they're close, the legacy ICE fleet is dragging.

In [31]:
# ---- Fleet turnover: sales share vs stock share, world ----

sales_share = slice_data(df, parameter="EV sales share",
                          region="World", mode="Cars", powertrain="EV") \
    .sort_values("year")
stock_share = slice_data(df, parameter="EV stock share",
                          region="World", mode="Cars", powertrain="EV") \
    .sort_values("year")

fig_turn = go.Figure()
for series, name, color in [(sales_share, "Sales share (new cars)", PALETTE["primary"]),
                             (stock_share, "Stock share (cars on road)", PALETTE["accent"])]:
    hist = series[series["category"]=="Historical"]
    proj = series[series["category"]=="Projection-STEPS"]
    fig_turn.add_trace(go.Scatter(
        x=hist["year"], y=hist["value"],
        mode="lines+markers", name=name,
        line=dict(color=color, width=3),
        marker=dict(size=8),
    ))
    if len(proj):
        bridge_x = [hist["year"].iloc[-1]] + proj["year"].tolist()
        bridge_y = [hist["value"].iloc[-1]] + proj["value"].tolist()
        fig_turn.add_trace(go.Scatter(
            x=bridge_x, y=bridge_y,
            mode="lines+markers", name=f"{name} (STEPS 2030)",
            line=dict(color=color, width=2, dash="dash"),
            marker=dict(size=10, symbol="diamond"),
            showlegend=True,
        ))

fig_turn.update_layout(
    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top",
               text="<b>The turnover gap races between sales share and stock share</b>"),
    height=500, xaxis_title="Year", yaxis_title="% of cars",
    legend=dict(orientation="h", y=-0.18, x=0.5, xanchor="center"),
    hovermode="x unified", margin=dict(t=60,b=60,l=0,r=500),
)
fig_turn.show()



**Tab 3 takeaways.**

- **Korea are nearly pure-BEV** (Norway, Iceland >85% BEV).
- **China is BEV-dominant** despite the size of its PHEV market, because BEV volume scaled faster.

---

## TAB 4: Socioeconomic & Policy

> *"Is EV adoption equitable across income levels, or only a rich-country story?"*

We classify countries into **High / Upper-middle / Lower-middle income** groups (World Bank-style heuristics), then plot **EV penetration (sales share)** vs **charging infrastructure density (chargers per million people)**.

In [32]:
# ---- Scatter: EV penetration vs charging-infra density, by income group ----

# Coarse income classification (manual; would normally pull from WB API)
INCOME = {
    # High income
    "Norway":"High","Sweden":"High","Iceland":"High","Denmark":"High","Finland":"High",
    "Netherlands":"High","Germany":"High","France":"High","United Kingdom":"High",
    "Switzerland":"High","Austria":"High","Belgium":"High","Ireland":"High",
    "Italy":"High","Spain":"High","Portugal":"High","Greece":"High","Luxembourg":"High",
    "United States":"High","Canada":"High","Australia":"High","New Zealand":"High",
    "Japan":"High","Korea":"High","Israel":"High","Estonia":"High","Slovenia":"High",
    "Czech Republic":"High","Slovakia":"High","Croatia":"High","Cyprus":"High",
    "Malta":"High","Latvia":"High","Lithuania":"High","Poland":"High","Hungary":"High",
    # Upper-middle income
    "China":"Upper-middle","Brazil":"Upper-middle","Mexico":"Upper-middle",
    "South Africa":"Upper-middle","Turkey":"Upper-middle","Turkiye":"Upper-middle",
    "Romania":"Upper-middle","Bulgaria":"Upper-middle","Costa Rica":"Upper-middle",
    "Colombia":"Upper-middle","Chile":"Upper-middle","Thailand":"Upper-middle",
    # Lower-middle income
    "India":"Lower-middle","Indonesia":"Lower-middle","Ukraine":"Lower-middle",
}

# Country population (millions, ~2023, approximate; for density normalization)
POP_M = {
    "Norway":5.5,"Sweden":10.5,"Iceland":0.4,"Denmark":5.9,"Finland":5.6,"Netherlands":17.9,
    "Germany":84.5,"France":68.0,"United Kingdom":68.0,"Switzerland":8.8,"Austria":9.0,
    "Belgium":11.7,"Ireland":5.1,"Italy":59.0,"Spain":48.0,"Portugal":10.3,"Greece":10.4,
    "Luxembourg":0.7,"United States":335.0,"Canada":40.0,"Australia":26.6,"New Zealand":5.2,
    "Japan":125.0,"Korea":51.7,"Israel":9.7,"Estonia":1.3,"Slovenia":2.1,"Czech Republic":10.5,
    "Slovakia":5.4,"Croatia":3.9,"Cyprus":1.3,"Malta":0.5,"Latvia":1.9,"Lithuania":2.8,
    "Poland":36.7,"Hungary":9.6,"China":1411.0,"Brazil":216.0,"Mexico":129.0,
    "South Africa":60.4,"Turkey":85.4,"Turkiye":85.4,"Romania":19.0,"Bulgaria":6.4,
    "Costa Rica":5.2,"Colombia":52.0,"Chile":19.6,"Thailand":71.7,"India":1428.0,
    "Indonesia":278.0,"Ukraine":36.7,
}

pen = slice_data(df, parameter="EV sales share",
                  mode="Cars", powertrain="EV",
                  category="Historical", year=LATEST_HIST,
                  agg_group="Country")
ch_density = total_chargers_by(df, ["region_country","year","category"]) \
    .query("category == 'Historical' and year == @LATEST_HIST")

socio = pen[["region_country","value"]].rename(columns={"value":"sales_share"})
socio = socio.merge(ch_density[["region_country","value"]].rename(columns={"value":"chargers"}),
                    on="region_country", how="left")
socio["income"]      = socio["region_country"].map(INCOME)
socio["pop_M"]       = socio["region_country"].map(POP_M)
socio["ch_per_M"]    = socio["chargers"] / socio["pop_M"]
socio = socio.dropna(subset=["income","pop_M","chargers"])
socio = socio[socio["chargers"] > 0].copy()

INCOME_COLORS = {"High":"#0B7A75","Upper-middle":"#F4A261","Lower-middle":"#E76F51"}

fig_soc = px.scatter(
    socio, x="ch_per_M", y="sales_share",
    color="income", color_discrete_map=INCOME_COLORS,
    size="pop_M", size_max=55,
    hover_name="region_country",
    labels={"ch_per_M":"Public chargers per million people",
            "sales_share":"EV sales share of new cars (%)",
            "income":"Income group"},
    title=f"<b>Adoption vs infrastructure density, {LATEST_HIST}</b>"
)
# annotate outliers (high adoption AT each income group)
for grp in ["High","Upper-middle","Lower-middle"]:
    grp_df = socio[socio["income"]==grp]
    if len(grp_df) == 0: continue
    top = grp_df.nlargest(2, "sales_share")
    for _, r in top.iterrows():
        fig_soc.add_annotation(
            x=r["ch_per_M"], y=r["sales_share"],
            text=r["region_country"], showarrow=True, arrowhead=2,
            ax=20, ay=-15, font=dict(size=10))

fig_soc.update_layout(height=600, margin=dict(t=60,b=60,l=0,r=500),
                      title=dict(x=0.1, y=0.95, xanchor="left", yanchor="top"),
                       legend=dict(orientation="h", y=-0.18, x=0.5, xanchor="center"))
fig_soc.show()



In [33]:
# summary stats by income
print("\nMedian EV sales share by income group:")
print(socio.groupby("income")["sales_share"].median().round(1).sort_values(ascending=False))


Median EV sales share by income group:
income
High            24.0
Upper-middle     7.4
Lower-middle     4.7
Name: sales_share, dtype: float64


---

## TAB 5: Forecasting

> *"The IEA STEPS gives us a 2030 endpoint. Can we triangulate that with our own statistical model?"*

We fit an **ARIMA(1,2,1)** to the historical world EV stock series, generate a forecast with 80% & 95% prediction intervals, and overlay the IEA STEPS scenario for comparison.

ARIMA is chosen over Prophet here because: (a) the series is short (~15 points) so simple wins, (b) we have no clear seasonality (annual data), and (c) statsmodels is a lighter dependency.

In [34]:
# ---- ARIMA forecast: world EV stock 2025-2030 ----

from statsmodels.tsa.arima.model import ARIMA

hist_stock = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World' and category == 'Historical'") \
    .sort_values("year").set_index("year")["value"]

# fit ARIMA(1,2,1) — second-difference captures the accelerating curve
model = ARIMA(hist_stock, order=(1, 2, 1))
res = model.fit()
print(res.summary().tables[1])

# forecast 6 steps ahead (2025..2030)
fc = res.get_forecast(steps=6)
fc_mean   = fc.predicted_mean
fc_ci_80  = fc.conf_int(alpha=0.20)
fc_ci_95  = fc.conf_int(alpha=0.05)
fc_mean.index   = range(LATEST_HIST, LATEST_HIST + 6)
fc_ci_80.index  = fc_mean.index
fc_ci_95.index  = fc_mean.index

# IEA STEPS 2030
steps_2030 = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World' and category == 'Projection-STEPS' and year == 2030") \
    ["value"].sum()

fig_fc = go.Figure()

# 95% CI band
fig_fc.add_trace(go.Scatter(
    x=list(fc_ci_95.index) + list(fc_ci_95.index)[::-1],
    y=list(fc_ci_95.iloc[:,1]/1e6) + list(fc_ci_95.iloc[:,0]/1e6)[::-1],
    fill="toself", fillcolor="rgba(244,162,97,0.18)",
    line=dict(color="rgba(0,0,0,0)"),
    name="95% prediction interval", showlegend=True,
    hoverinfo="skip",
))
# 80% CI band
fig_fc.add_trace(go.Scatter(
    x=list(fc_ci_80.index) + list(fc_ci_80.index)[::-1],
    y=list(fc_ci_80.iloc[:,1]/1e6) + list(fc_ci_80.iloc[:,0]/1e6)[::-1],
    fill="toself", fillcolor="rgba(244,162,97,0.35)",
    line=dict(color="rgba(0,0,0,0)"),
    name="80% prediction interval", showlegend=True,
    hoverinfo="skip",
))
# historical
fig_fc.add_trace(go.Scatter(
    x=hist_stock.index, y=hist_stock.values/1e6,
    mode="lines+markers", name="Historical (IEA)",
    line=dict(color=PALETTE["primary"], width=3),
    marker=dict(size=7),
    hovertemplate="%{x}: %{y:.1f} M EVs<extra></extra>",
))
# ARIMA point forecast
fig_fc.add_trace(go.Scatter(
    x=fc_mean.index, y=fc_mean.values/1e6,
    mode="lines+markers", name="ARIMA forecast (mean)",
    line=dict(color=PALETTE["accent"], width=3, dash="dash"),
    marker=dict(size=8, symbol="diamond"),
    hovertemplate="%{x}: %{y:.1f} M EVs (forecast)<extra></extra>",
))
# IEA STEPS 2030 marker
fig_fc.add_trace(go.Scatter(
    x=[2030], y=[steps_2030/1e6],
    mode="markers+text", name="IEA STEPS 2030",
    marker=dict(size=18, color=PALETTE["danger"], symbol="star",
                line=dict(color="white", width=2)),
    text=[f"  IEA STEPS: {steps_2030/1e6:.0f} M"],
    textposition="middle right",
    hovertemplate="IEA STEPS 2030: %{y:.0f} M EVs<extra></extra>",
))

fig_fc.update_layout(
    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top", text="<b>World EV car stock - ARIMA forecast vs IEA STEPS</b>"),
    height=520, xaxis_title="Year",
    yaxis_title="EV cars on the road (millions)",
    hovermode="x unified",
    legend=dict(orientation="h", x=0.4, y=1, xanchor="left", yanchor="top"),
    margin=dict(l=20,r=200,t=80,b=70),
)
fig_fc.show()

print(f"\nARIMA 2030 forecast mean : {fc_mean.iloc[-1]/1e6:.1f} M EVs")
print(f"ARIMA 2030 80% interval  : {fc_ci_80.iloc[-1,0]/1e6:.1f} – {fc_ci_80.iloc[-1,1]/1e6:.1f} M")
print(f"IEA STEPS 2030           : {steps_2030/1e6:.1f} M EVs")


                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          1.0000      0.263      3.809      0.000       0.485       1.515
ma.L1          0.3335      0.282      1.183      0.237      -0.219       0.886
sigma2      7.412e+11   1.12e-13   6.59e+24      0.000    7.41e+11    7.41e+11


d:\Data_Projects\data-visualization-shiny\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Data_Projects\data-visualization-shiny\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Data_Projects\data-visualization-shiny\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Data_Projects\data-visualization-shiny\venv\Lib\site-packages\statsmodels\tsa\b


ARIMA 2030 forecast mean : 246.2 M EVs
ARIMA 2030 80% interval  : 207.7 – 284.7 M
IEA STEPS 2030           : 232.2 M EVs


**Tab 5 takeaways.**

- Our ARIMA and the IEA STEPS scenario are **in the same order of magnitude**, with ARIMA tending to be more aggressive when projected from 2024 alone (because it has no policy ceiling baked in). The IEA STEPS is policy-grounded; ARIMA is trend-grounded.

---

## Conclusions

1. **The hockey stick is real but the fleet lags the market.** EV sales-share is racing toward 20–25% globally while stock share is still ~5–6%. The 2020s are the *sales transition*; the 2030s will be the *fleet transition*.

2. **Two adoption archetypes coexist.** Norway-style markets win on intensity (saturated, BEV-dominant, high charger density). China-style markets win on volume (state-coordinated infrastructure + manufacturer scale). They are not the same playbook.

3. **Charging infrastructure is, globally, slightly behind EV deployment in 2024.** Several large markets have charger-to-EV ratios well above the comfort threshold — a real risk for new entrants and a real opportunity for charging-network operators.

4. **BEV is winning the long game**, but PHEV remains meaningful in markets with weak fast-charging networks or strong subsidies for them. As BEV TCO crosses ICE parity (likely 2026–28), PHEV share will compress.

5. **2030 STEPS is aggressive but not radical.** ARIMA-style trend extrapolation sits in the same neighborhood. The real risk to 2030 targets is **fleet turnover**, not sales penetration. Scrappage incentives and corporate fleet electrification are the high-leverage levers.

6. **Oil displacement compounds faster than the public realizes** — even at a conservative CAGR, the EV fleet displaces ~5–10 Mbd by 2030 in our extrapolation, which is structurally bearish for refined-product demand growth and bullish for grid investment.

---

### What we'd build next (analyst's note)

- **Cohort analysis by vintage** — track 2018 / 2020 / 2022 cohorts to estimate real fleet retirement curves rather than assume them.
- **Bottleneck modelling** — overlay battery-mineral supply forecasts (Li, Co, Ni, graphite) on the stock forecast.
- **Charger utilization** — chargers per EV is a stock metric; **kWh per charger per day** would tell us how stressed the network actually is.
- **Policy event-study** — formal causal inference around big policy events (IRA, China NEV mandate, EU 2035 ICE ban) instead of the eyeball method used here.
